# 2.0 — Teste user-item (MAP / recall)

**Objetivo:** avaliar se o ranking do `Recommender` (embedding + reorder LLM opcional) recupera procedimentos (`catalog_id`) que o paciente fez **após** `reference_date`.

**Ajuste antes de rodar:** `PARTNER`, `reference_date`, `individual_id`, campos de embedding; parquet em `LOCAL_PATH`.

**Fluxo:** baseline MAP/recall → `reorder_ranking` → `eval_recommender` em lote. Ver [notebooks/README.md](../README.md).

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
from IPython.display import display, Markdown

from wokibi_eval.evaluation.ranking import mean_average_precision_at_k, recall_at_k

from wokibi_ai.utils.string_utils import clean_text_to_json
from wokibi_data.config import LOCAL_PATH, CLIENT_ID_COLUMN, EVENT_DATE

from wokibi_data.beans.mongo.individual import IndividualBean
from wokibi_data.marts.events import Events

from wokibi_ai.models.recommender.recommender import Recommender
from wokibi_eval.recommender.evaluation import eval_recommender


### Racional
- Verifica a efetiviade do ranking de procedimentos feitos no futuro para os infivíduos definidos na base de treino.

In [ ]:
PARTNER = "nefrologia"
reference_date = "2026-01-01"

embedding_user = "embeddings_avg"
embedding_item = "embedding_gpt"

# amazon: 110001
# nefrologia: 11912169

individual_id = 11912169

In [ ]:
recommender = Recommender(PARTNER, k=50)

In [ ]:
DUCKDB_QUERY = f"""
    SELECT * FROM '{LOCAL_PATH}internal_data/{PARTNER}/events.parquet' ORDER BY {EVENT_DATE} ASC
"""
location_duckdb = {
    "type": "duckdb", "query": DUCKDB_QUERY,
}
events = Events(PARTNER)
events.load_data(location=location_duckdb)
events.transform(new_features=False, cast_columns=False)
events.table.shape, events.table.transaction_id.nunique()

In [ ]:
individualBean = IndividualBean(PARTNER)
individualBean.get_bean_list_from_mongo(PARTNER)
customers = individualBean.to_df()
customers.shape, customers.id.nunique()

In [ ]:
df_events = events.table[events.table[CLIENT_ID_COLUMN].isin(customers.id)]
df_events = df_events.query(f"{EVENT_DATE} > '{reference_date}'")
df_events.shape, df_events[CLIENT_ID_COLUMN].nunique()

In [ ]:
df_events.sample()

In [ ]:
user, ids = recommender.itens_by_user(individual_id)
user.keys(), len(user['items']), user['items'], len(ids)

In [ ]:
# test dataset
test_items = df_events[df_events[CLIENT_ID_COLUMN] == individual_id]["catalog_id"].unique()
len(test_items), test_items

In [ ]:
predict_items = np.array([p for p in ids if p not in user['items']])
test_items, predict_items

In [ ]:
recall_at_k(test_items, predict_items), mean_average_precision_at_k(test_items, predict_items)

In [ ]:
selected_itens = recommender.get_data_by_ids(predict_items.tolist(), "item")
selected_itens[0]

In [ ]:
paramenters = {
    "ALVO": user['history'],
    "CANDIDATOS": selected_itens,
}
r = recommender.reorder_ranking(paramenters)
display(Markdown(r))

In [ ]:
r = clean_text_to_json(r)
len(r['item_ids'])

In [ ]:
recall_at_k(test_items, r['item_ids']), mean_average_precision_at_k(test_items, r['item_ids'])

In [ ]:
memory = {}

In [ ]:
result_memory = eval_recommender(memory, recommender, df_events, PARTNER, "id")
result_memory.keys()

In [ ]:
memory.keys()

In [ ]:
result_memory.keys()

In [ ]:
result_memory["y_true_geral"][0], result_memory["y_pred_geral"][0]

In [ ]:
recall_at_k(result_memory["y_true_geral"], result_memory["y_pred_geral"]), recall_at_k(result_memory["y_true_geral"][0], result_memory["y_pred_geral"][0])

In [ ]:
mean_average_precision_at_k(result_memory["y_true_geral"], result_memory["y_pred_geral"]), mean_average_precision_at_k(result_memory["y_true_geral"][0], result_memory["y_pred_geral"][0])

In [ ]:
np.mean(result_memory["users_recalls"]), np.mean(result_memory["users_maps"])